# Parquet and DuckDB Primer

This cookbook makes extensive use of `duckdb` and a file format called Parquet (specifically GeoParquet). This notebook serves to get you familiar with the technology, why it's being used, some more advanced usage, and some alternatives.

## The What and Why of (Geo)Parquet

As noted in the Parquet [documentation](https://parquet.apache.org/docs/overview/): "Apache Parquet is an open source, column-oriented data file format designed for efficient data storage and retrieval."

In plain language, this means that Parquet is a modern, performant alternative to the plain text CSV (Comma Separated Variables) format. The following code cell pulls in some example CSV-formatted data from the [Iowa Environmental Mesonet](https://mesonet.agron.iastate.edu/) (IEM) and contains the following fields: station (Denver International Airport), day (Jan 1-10, 2026), and max recorded temperature in Fahrenheit.

In [1]:
import requests

iem_url = "https://mesonet.agron.iastate.edu/cgi-bin/request/daily.py"
params = {
    "network": "CO_ASOS", "stations": "DEN",
    "year1": "2026", "month1": "1", "day1": "1",
    "year2": "2026", "month2": "1", "day2": "10",
    "var": "max_temp_f",
    "na": "blank",
    "format": "csv"
}
den_data = requests.get(
    iem_url,
    params=params
)
print(den_data.content.decode())

station,day,max_temp_f
DEN,2026-01-01,64.0
DEN,2026-01-02,60.0
DEN,2026-01-03,64.0
DEN,2026-01-04,67.0
DEN,2026-01-05,61.0
DEN,2026-01-06,57.0
DEN,2026-01-07,58.0
DEN,2026-01-08,41.0
DEN,2026-01-09,32.0
DEN,2026-01-10,44.0



For small data sets such as the above, this is an acceptable file format. However, consider a dataset with these properties:

- many stations located globaly
- a large number of data fields
- collected hourly
- decades-worth of data

Such a file in a CSV format would quickly become unwieldy. Furthermore, much of the data is "repeated." That is, all the Denver station data has "DEN" in the station column. Over hundreds, or thousands, of data entries this redundant data can add up to unnecessarily large file size. In addition, this particular example does not have geospatial data.

With Parquet, this data can be saved in a more storage-efficient format. GeoParquet extends Parquet to add geospatial/geometry data and metadata to Parquet data sets. An interested reader can read the introductory blog post with more detail [here](https://xebia.com/blog/introducing-the-geoparquet-data-format/).

The IEM provides a historical archive of global ASOS data in the native METAR format. To conclude our GeoParquet story, [Dynamical.org](https://dynamical.org/catalog/asos-parquet/) has created a processing pipeline that takes raw METAR from the IEM, transforms it into the GeoParquet format, and makes it readily available via the internet.

## The What and Why of DuckDB

### How to Construct a Query